# Session 3: Exercises

**Do these before session 4.** They take about ninety minutes; the ⭐⭐⭐ ones join the session's ideas
together.

Same pattern as last week: **predict, then measure.** Write down what you expect before running a cell.
The gap between your prediction and the result shows you what you actually believed. Predictions and
answers go in the cells marked **✍️**; code to write is marked `# TODO`.

Worked solutions are released after the next session.

## Setup (just run it)

The client with retries, the inbox, `as_block`, `ask_schema`, and the v0.2 schemas.

In [ ]:
import json
import re
import sys
from collections import Counter
from pathlib import Path
from typing import Literal

import yaml
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError, ValidationInfo, field_validator

sys.path.insert(0, str(Path.cwd() / "project"))
from config import settings                                          # noqa: E402
from schemas import BANGLA_DIGITS, ORDER_NUMBER, TicketFacts, TriageResult   # noqa: E402
from triage import triage_system                                     # noqa: E402

client = genai.Client(http_options=types.HttpOptions(
    retry_options=types.HttpRetryOptions(attempts=4, http_status_codes=[429, 500, 503])))
MODEL = settings.MODEL
RAW_TICKETS = yaml.safe_load((settings.DATA_DIR / "tickets.yml").read_text(encoding="utf-8"))
by_id = {t["id"]: t for t in RAW_TICKETS}


def as_block(t: dict) -> str:
    return f"<ticket>\nSubject: {t['subject']}\n{t['body']}\n</ticket>"


def ask_schema(schema, contents, system=None):
    '''One structured call; returns the whole response (lesson section 2).'''
    config = {"response_mime_type": "application/json", "response_schema": schema}
    if system:
        config["system_instruction"] = system
    return client.models.generate_content(model=MODEL, contents=contents, config=config)


print(f"{MODEL} · {len(RAW_TICKETS)} tickets")

## ⭐ Exercise 1: Read the error before you run it

**The idea.** Lesson section 1.3. A `ValidationError` is a list of structured problems, and reading
one quickly is a skill you'll use every day. Here's a ticket with several things wrong.

**✍️ Predict first:** how many errors will Pydantic report, and for each one, which field (`loc`) and
which `type`? Write them down before you run the cell.

**✍️ Your prediction:**

*(write it here before you run the next cell)*

In [ ]:
from schemas import Ticket

bad = {"id": "TCK-1", "subject": "help", "body": "", "from": "a@example.com",
       "category": "Shipping", "received": "yesterday"}

# TODO: validate `bad` with Ticket.model_validate and print loc, type and msg for every error.

**✍️ Your answer.** How many did you predict? Which one surprised you?

*(your answer here)*

## ⭐ Exercise 2: The confidence field

**The idea.** It's tempting to ask the model how sure it is. Add a `confidence` field to triage and
run it on an ambiguous ticket and an obvious one, three times each.

**✍️ Predict first:** will the ambiguous ticket (`TCK-003`, an angry "where is my order") get lower
confidence than the obvious one (`TCK-016`, "please cancel ORD-5002")?

**✍️ Your prediction:**

*(write it here before you run the next cell)*

In [ ]:
# TODO: define WithConfidence: TriageResult's category field, plus
#       confidence: Literal["low", "medium", "high"], declared after the category.
# Then run it three times on each ticket and print category and confidence.

**✍️ Your answer.** Did confidence track how hard the ticket is? Would you ship this field?

*(your answer here)*

## ⭐⭐ Exercise 3: Tracking numbers

**The idea.** Lesson sections 3.3 and 3.4, on a new field. Courier tracking numbers look like
`SW-88231` (TCK-017 has one). Add `tracking_numbers` to a facts model with the same pair of validators:
normalise first, then check against the ticket. Then score it over the inbox: found, missed, invented.

**✍️ Predict first:** how many tracking numbers are in the inbox, and will the model invent any?

**✍️ Your prediction:**

*(write it here before you run the next cell)*

In [ ]:
TRACKING = re.compile(r"^(SW)?[\s\-#]*(\d+)$", re.IGNORECASE)


class Tracking(BaseModel):
    tracking_numbers: list[str] = Field(description="")    # TODO: describe it, like TicketFacts.order_ids

    # TODO: a mode="before" validator that turns "sw 88231" or "SW-৮৮২৩১" into "SW-88231"
    #       and rejects anything that isn't shaped like a tracking number.
    # TODO: an after validator that raises if a number isn't in the ticket text from the context.
    #       Return early when info.context is None: the SDK runs your validators too, without
    #       context, when it fills .parsed (lesson section 3.4).


def score_tracking():
    found = missed = invented = 0
    for t in RAW_TICKETS:
        text = f"{t['subject']}\n{t['body']}"
        truth = {f"SW-{n}" for n in re.findall(r"SW-(\d+)", text.translate(BANGLA_DIGITS))}
        r = ask_schema(Tracking, as_block(t))
        try:
            got = set(Tracking.model_validate_json(r.text, context={"ticket": text}).tracking_numbers)
        except ValidationError:
            got = set()
        found += len(got & truth)
        missed += len(truth - got)
        invented += len(got - truth)
    print(f"found {found} · missed {missed} · invented {invented}")


score_tracking()

**✍️ Your answer.** What did the score show, and what would the score have hidden if you'd averaged missed and invented into one accuracy number?

*(your answer here)*

## ⭐⭐ Exercise 4: The limit that usually works

**The idea.** Lesson section 2, step 4. Measure how often `.parsed` comes back `None` for three caps
on `summary` (40, 60 and 80 characters), each with a description that **agrees** with the cap and one
that **disagrees** ("under 20 words").

**✍️ Predict first:** which of the six combinations fails most? Does any reach zero?

**✍️ Your prediction:**

*(write it here before you run the next cell)*

In [ ]:
from pydantic import create_model


def none_rate(cap: int, description: str, tickets=RAW_TICKETS[:10]) -> int:
    '''How many of `tickets` come back with .parsed None for this cap and description.'''
    Schema = create_model("Schema", summary=(str, Field(max_length=cap, description=description)))
    # TODO: run ask_schema(Schema, as_block(t)) for each ticket and count .parsed is None
    return 0


# TODO: for cap in (40, 60, 80), print none_rate with an agreeing description
#       (e.g. f"One sentence, at most {cap} characters.") and with "One sentence, under 20 words."

**✍️ Your answer.** What made failures rarer, and what would make them *handled*?

*(your answer here)*

## ⭐⭐ Exercise 5: Earn your examples

**The idea.** Lesson section 4.1. The shipped examples bought nothing. Write three new ones aimed at the
tickets the rules still miss (the `complaint` vs `order_status` and `return` vs `warranty` borderlines),
then compare rules-only with rules-plus-your-examples, **twice each**.

Write your examples fresh. Don't copy inbox tickets: an example in the prompt is never a test case.

**✍️ Predict first:** will targeted examples beat the rules by more than the noise (about ±2 tickets)?

**✍️ Your prediction:**

*(write it here before you run the next cell)*

In [ ]:
def score(system: str) -> int:
    hits = 0
    for t in RAW_TICKETS:
        r = ask_schema(TriageResult, as_block(t), system)
        if r.parsed is not None and r.parsed.category == t["category"]:
            hits += 1
    return hits


MY_EXAMPLES = '''
'''   # TODO: three <ticket>...</ticket> + category: ... examples, written fresh

rules = triage_system(with_examples=False)
# TODO: print score(rules) twice and score(rules + MY_EXAMPLES) twice

**✍️ Your answer.** Was the difference bigger than the run-to-run spread? What would you ship?

*(your answer here)*

## ⭐⭐⭐ Exercise 6: Code decides the queue

**The idea.** "The model advises, code decides", with no model call at all. The lesson wrote the CTO's
spreadsheet to `triage.csv`. Give urgency a service-level target with an `Enum` (lesson appendix A),
then produce the queue the support team works from: soonest SLA first, and within the same SLA,
tickets inside the return window before unknown ones before those outside.

**✍️ Predict first:** which ticket ends up at the top, and why?

**✍️ Your prediction:**

*(write it here before you run the next cell)*

In [ ]:
import csv
from enum import Enum

with open("triage.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))


class Urgency(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    # TODO: an sla_hours property: high 4, medium 24, low 72


class QueueRow(BaseModel):
    id: str
    urgency: Urgency
    needs_human: bool
    return_window: Literal["inside", "unknown", "outside"]
    summary: str


# TODO: validate every row into a QueueRow (note: needs_human arrives as the string "True"),
#       sort by (sla_hours, return-window order), and print the queue.

**✍️ Your answer.** Which parts of that queue came from the model, and which from code? Where would you change the SLA if the manager asked?

*(your answer here)*

## ⭐⭐⭐ Exercise 7: When does caching start?

**The idea.** Lesson section 5. Find the smallest prefix that gets an implicit cache hit on your key,
by growing a library of policy documents and asking the same question three times at each size. Then
estimate how large v0.2's system slot would have to grow before caching mattered.

**✍️ Predict first:** at roughly how many tokens does the first hit appear?

**✍️ Your prediction:**

*(write it here before you run the next cell)*

In [ ]:
policy = Path("../session-02/returns_policy.md").read_text(encoding="utf-8")


def library(n_docs: int) -> str:
    return "\n".join(f"<document index='{i}'>\n{policy.replace('ShopWise', f'Store{i}')}\n</document>"
                     for i in range(n_docs))


def cached_on_repeat(system: str) -> int | None:
    '''Ask the same question three times; return what the third call reports as cached.'''
    # TODO: three generate_content calls with config={"system_instruction": system};
    #       return the last call's usage_metadata.cached_content_token_count
    return None


# TODO: for n in (4, 8, 12, 16, 20, 24): print the token count of library(n) and cached_on_repeat(library(n))

**✍️ Your answer.** Where did caching start, and what does that mean for v0.2 and for session 2's date-first decision?

*(your answer here)*